# **Путь, а не только точка: Blur IG и Guided IG**

Практика к модулю [«Атрибуция от аксиом»](https://open-xai-platform.web.app).

Урок говорит важную вещь: у интеграла по пути **две ручки** — откуда идем (baseline) и как идем
(путь). Expected Gradients крутят первую, Guided IG только вторую, Blur IG меняет обе.

Отсюда следствие, которое легко проговорить и невозможно запомнить без числа:
**Guided IG слепое пятно черного baseline не чинит** — он baseline вообще не трогает.
Сейчас увидим это в процентах.

Считает на процессоре пару минут: три метода по 32 шага.

In [ ]:
import io
import urllib.request

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights

torch.manual_seed(0)
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main'
STEPS = 32                                                  # шагов интегрирования — хватает, чтобы полнота сошлась в пределах процента
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).eval()
for p in model.parameters():
    p.requires_grad_(False)

tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
raw = urllib.request.urlopen(f'{DATA}/data/cat.jpg', timeout=30).read()
img = tf(Image.open(io.BytesIO(raw)).convert('RGB')).unsqueeze(0)     # картинка в [0,1]: нормировка делается внутри logit


def logit(x, c):
    """Логит класса. Нормировка внутри, чтобы путь строить в пространстве картинки."""
    return model((x - MEAN) / STD)[0, c]


def grad(x, c):
    x = x.clone().requires_grad_(True)
    logit(x, c).backward()
    return x.grad


CLS = int(model((img - MEAN) / STD).argmax())
print(f'класс: {CLS}')

## 1. Как выглядит путь Blur IG

Обычный IG идет по прямой в пространстве пикселей: от черного изображения к нашему, линейно.
Blur IG идет иначе — от сильно размытого снимка к четкому, постепенно уменьшая радиус
размытия. Посмотрим на этот путь глазами.

In [ ]:
def blur(x, sigma):
    """Гауссово размытие радиуса sigma: два одномерных прохода вместо одного двумерного."""
    if sigma < 0.1:
        return x.clone()
    k = int(4 * sigma) | 1
    t = torch.arange(k, dtype=torch.float32) - k // 2
    g = torch.exp(-t ** 2 / (2 * sigma ** 2))
    g = (g / g.sum()).view(1, 1, -1)
    y = F.conv2d(F.pad(x, (k // 2,) * 2 + (0, 0), mode='reflect'),
                 g.expand(3, 1, 1, k), groups=3)
    return F.conv2d(F.pad(y, (0, 0) + (k // 2,) * 2, mode='reflect'),
                    g.view(1, 1, -1, 1).expand(3, 1, k, 1), groups=3)


fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, s in zip(axes, (20.0, 10.0, 5.0, 2.0, 0.0)):
    ax.imshow(blur(img, s)[0].permute(1, 2, 0).numpy())
    ax.set_title(f'sigma = {s:.0f}')
    ax.axis('off')
plt.tight_layout()
plt.show()

Обратите внимание: **на всем пути картинка остается картинкой.** Это и есть
довод в пользу такого пути. У прямого пути середина — полупрозрачная серая мазня, которой
в природе не бывает, и градиенты там шумные; у пути через размытие каждая промежуточная точка
похожа на настоящее изображение, просто снятое не в фокусе.

In [ ]:
def ig(x, baseline, c, steps=STEPS):
    """Обычный IG: прямой путь от baseline к объекту."""
    total = torch.zeros_like(x)
    for k in range(1, steps + 1):
        total += grad(baseline + (k / steps) * (x - baseline), c)
    return (x - baseline) * total / steps


def blur_ig(x, c, sigma_max=20.0, steps=STEPS):
    """Blur IG: путь идет от сильно размытого снимка к четкому, через уменьшение sigma."""
    sigmas = torch.linspace(sigma_max, 0.0, steps + 1)
    total, prev = torch.zeros_like(x), blur(x, float(sigmas[0]))
    for s in sigmas[1:]:
        cur = blur(x, float(s))
        total += grad(0.5 * (prev + cur), c) * (cur - prev)      # градиент в середине отрезка, умноженный на сдвиг по пути
        prev = cur
    return total


def guided_ig(x, baseline, c, steps=STEPS, q=0.5):
    """Guided IG: на каждом шаге двигаются только признаки с наименьшим по модулю градиентом."""
    cur, total = baseline.clone(), torch.zeros_like(x)
    for k in range(steps):
        g = grad(cur, c)
        remaining = baseline + ((k + 1) / steps) * (x - baseline) - cur
        mask = (g.abs() <= torch.quantile(g.abs().flatten(), q)).float()   # нижние q процентов по модулю градиента — там функция меняется спокойнее
        step = remaining * mask + remaining * (1 - mask) * (k + 1 == steps)
        total += g * step
        cur = cur + step
    return total


zero = torch.zeros_like(img)
a_ig, a_blur, a_guided = ig(img, zero, CLS), blur_ig(img, CLS), guided_ig(img, zero, CLS)
print(f'посчитано')

## 2. Полнота: сошлось ли

Все три метода — интегралы по пути, и у всех трех сумма атрибуций обязана равняться разности
логитов между концами пути. Проверим.

In [ ]:
f_x = logit(img, CLS).item()
for name, a, ref in (('IG', a_ig, zero),
                     ('Blur IG', a_blur, blur(img, 20.0)),
                     ('Guided IG', a_guided, zero)):
    print(f'   {name:11} сумма атрибуций {a.sum():+7.3f}   f(x) - f(точка отсчета) '
          f'{f_x - logit(ref, CLS).item():+7.3f}')

Сходится у всех трех, с точностью в пределах процента — это ошибка суммы Римана
при 32 шагах, и она уменьшается с ростом их числа.

**Смотрите на точку отсчета в третьей колонке.** У IG и Guided IG она нулевая, у Blur IG —
размытый снимок. Числа поэтому разные, и это не расхождение, а **разные вопросы**: «почему
не пустота» против «что дает детализация сверх общей композиции». Ровно то, о чем урок говорит
в разделе про baseline.

**Задание 1.** Посчитайте полноту при 8, 32 и 128 шагах и посмотрите, как убывает невязка.
Убывает ли она монотонно?

In [ ]:
# Ваш код здесь

## 3. Главное: слепое пятно чинит только один из двух

Урок объясняет, что у черного baseline есть слепое пятно: пиксель, совпавший с baseline,
получает строго нулевую атрибуцию, потому что множитель $(x_i - x'_i)$ обнуляет все. Значит
темные области изображения выпадают из объяснения.

Померим это. Возьмем темные пиксели и посмотрим, какая доля модуля атрибуции на них
приходится.

In [ ]:
dark = (img.mean(1, keepdim=True) < 0.25).float()
print(f'темные области занимают доли площади снимка: {dark.mean().item() * 100:.1f} %\n')
for name, a in (('IG', a_ig), ('Blur IG', a_blur), ('Guided IG', a_guided)):
    mass = a.abs().sum(1, keepdim=True)
    print(f'   {name:11} {(mass * dark).sum().item() / mass.sum().item() * 100:5.1f} %')

Вот ради этой таблицы все и затевалось.

- **IG с нулевым baseline** отдал темным областям около полутора процентов атрибуции при
  их доле площади в разы больше. Слепое пятно налицо — ровно то, что описывает урок.
- **Blur IG** отдал им больше их доли площади. Пятна нет вовсе: на пути через размытие
  темный пиксель ни в один момент не совпадает с точкой отсчета, потому что точка отсчета —
  размытая версия его самого, а не черный цвет.
- **Guided IG** почти не сдвинулся с места. **И это правильно, а не поломка.** Guided IG
  меняет путь, а baseline у него остался нулевым — значит и слепое пятно осталось. Метод
  чинит другую беду.

**Задание 2.** Замените в `guided_ig` нулевой baseline на размытый снимок и повторите
измерение. Исчезнет ли пятно? Что это говорит о том, какая из двух ручек за него отвечает?

In [ ]:
# Ваш код здесь

## 4. У Blur IG слепое пятно тоже есть — просто в другом месте

Прежде чем идти дальше, стоит проверить утверждение «Blur IG чинит слепое пятно» на прочность.
Пятно у чёрного baseline берётся из множителя, обнуляющего вклад там, где вход совпал с точкой
отсчёта. У Blur IG точка отсчёта другая — размытая версия самого снимка. Спросим: **бывает ли,
что пиксель совпадает и с ней?**

Бывает: в середине большой однородной области размытие почти ничего не меняет. Значит на всём
пути значение такого пикселя стоит на месте, а вклад считается по сдвигу вдоль пути — и вклада
почти не будет.

Померим: возьмём пиксели, у которых снимок и его размытая версия близки, и посмотрим на их долю
атрибуции.

In [ ]:
same = (img - blur(img, 20.0)).abs().mean(1, keepdim=True)
for thr in (1e-3, 1e-2):
    mask = (same < thr).float()
    mass = a_blur.abs().sum(1, keepdim=True)
    print(f'   порог {thr:g}: доля площади {mask.mean().item() * 100:5.2f} %   '
          f'доля атрибуции {(mass * mask).sum().item() / mass.sum().item() * 100:6.3f} %')

Доля атрибуции у таких пикселей **в разы меньше их доли площади** — то же самое,
что мы видели у чёрного baseline на тёмных областях, только область другая.

Ровно нуля здесь не выходит, и это тоже объяснимо: размытие конечного радиуса никогда не
оставляет пиксель совсем неизменным, поэтому множитель мал, но не ноль. У чёрного baseline
в чёрном пикселе он обнуляется точно.

**Вывод, который стоит унести вместо лозунга «Blur IG лучше»:** слепое пятно есть у любого
метода, который определяет важность через сравнение с эталоном. Вопрос не в том, чтобы найти
метод без пятна, а в том, чтобы знать, **где** оно у вашего, и совпадает ли это место с тем,
что вам важно на снимке.

## 5. А что тогда чинит Guided IG

Свою беду: шум. Прямой путь проходит через полупрозрачные картинки, которых сеть не видела,
градиенты там скачут, и этот скач оседает в карте крапом. Guided IG на каждом шаге двигает
только те признаки, у которых градиент по модулю мал, — то есть идет там, где функция меняется
спокойнее.

Померим шумность как средний перепад между соседними пикселями карты.

In [ ]:
for name, a in (('IG', a_ig), ('Blur IG', a_blur), ('Guided IG', a_guided)):
    m = a.abs().sum(1, keepdim=True)
    m = m / m.max()
    tv = ((m[:, :, 1:] - m[:, :, :-1]).abs().mean() + (m[..., 1:] - m[..., :-1]).abs().mean())
    print(f'   {name:11} {tv.item():.5f}')

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (name, a) in zip(axes, (('IG', a_ig), ('Blur IG', a_blur), ('Guided IG', a_guided))):
    m = a.abs().sum(1)[0]
    ax.imshow((m / m.quantile(0.99)).clamp(0, 1).numpy(), cmap='inferno')
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

Guided IG заметно тише обычного IG — примерно вдвое. Blur IG тоже тише, но по
другой причине: его путь идет через размытые изображения, и низкочастотность пути передается
карте.

**Итог, который стоит унести целиком.** Две ручки — два разных лекарства:

| что меняем | метод | что чинит | что НЕ чинит |
| --- | --- | --- | --- |
| точку отсчета | Expected Gradients | слепое пятно | шум пути |
| путь | Guided IG | шум | слепое пятно |
| и то и другое | Blur IG | пятно тёмных областей и шум | своё пятно на однородных областях |

**Задание 3.** Постройте карту Expected Gradients (код — в тетради «Expected Gradients» этого
модуля) и добавьте её в обе таблицы, слепого пятна и шумности. Встанет ли она туда, куда
обещает строка про неё? И отдельно: есть ли слепое пятно у неё самой, и если да — где?

In [ ]:
# Ваш код здесь

## Что унести из тетради

- **У интеграла по пути две независимые ручки**, и путать их дорого: Guided IG не заменяет
  Expected Gradients, а Blur IG не «просто получше IG».
- **Слепое пятно измеримо в процентах**, а не только описуемо словами. Если ваши объекты
  бывают темными, эту долю стоит посчитать до того, как поверить карте.
- **Полнота у всех трех держится**, но отклонение считается от разных точек отсчета. Число
  из отчета без указания точки отсчета не значит ничего.
- **На пути Blur IG каждая промежуточная точка похожа на настоящее изображение.** Это довод
  не эстетический: градиенты вне распределения данных шумят, и весь этот шум честно попадает
  в интеграл.